# PeatWatch U-Net training on Colab (M9)

Runbook: `docs/runbooks/colab.md`. Run the cells top to bottom.

Before you start:
1. Runtime > Change runtime type > **T4 GPU** (free tier is enough).
2. Upload `peatwatch_data.zip` (made by `docs/runbooks/colab.md` step 2) to Google Drive at `MyDrive/peatwatch/peatwatch_data.zip`. It holds `composites/*.tif`, `labels/*.tif` and `MANIFEST.json`.

Everything the run produces goes to `MyDrive/peatwatch/unet/` (checkpoints, `best.pt`, `split.json`, `config.json`) and `MyDrive/peatwatch/train.log`, so a disconnect loses nothing. If the session dies, rerun sections 1 to 5 and then the training cell: it resumes from `last.pt`.

## 1. Settings

In [ ]:
# ruff: noqa
# Edit here, then run.
DRIVE_DIR = "/content/drive/MyDrive/peatwatch"  # data zip in, results out
REPO_URL = "https://github.com/22KarthikV/Peatland.git"
BRANCH = "main"  # must contain ml/src/ml/train.py (M8) and scripts/check_artifact.py
EPOCHS = None  # None keeps the value from ml/configs/unet.yaml (50)
BATCH_SIZE = 8  # lower to 4 if CUDA runs out of memory
PATCH_SIZE = 128  # 256 px (2.56 km) patches rarely fit in one 5 km split block: most get dropped
STRIDE = 64  # 50 % overlap; the spatial split keeps overlapping patches in the same split
NUM_WORKERS = 2
USE_MLFLOW = False  # SQLite on the Drive mount can lock up; the log file has the metrics

## 2. GPU, Python, Drive

In [ ]:
import subprocess, sys, time

print(sys.version)
print(
    subprocess.run(
        "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
        shell=True,
        capture_output=True,
        text=True,
    ).stdout
    or "NO GPU: switch the runtime to T4 GPU"
)
import torch

assert torch.cuda.is_available(), (
    "No CUDA device. Runtime > Change runtime type > T4 GPU, then rerun."
)
print("torch", torch.__version__, "cuda", torch.version.cuda)

from google.colab import drive

drive.mount("/content/drive")
import os

os.makedirs(DRIVE_DIR, exist_ok=True)
print(os.listdir(DRIVE_DIR))

## 3. Clone the repo and install

`pip install -e ./ml` is enough for training (the `pipeline` package is only needed to build composites). Do **not** run `uv sync --extra train` here: the lockfile pins CPU-only torch wheels, and Colab already has a CUDA build.

The repo asks for Python 3.13. If this Colab runtime is older, the cell passes `--ignore-requires-python`; if pip cannot resolve the pins it falls back to `--no-deps` plus the packages that are really imported.

In [ ]:
import os, subprocess, sys


def sh(cmd):
    print("$", cmd)
    return subprocess.run(cmd, shell=True).returncode


if not os.path.isdir("/content/Peatland"):
    assert sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/Peatland") == 0
else:
    sh(f"git -C /content/Peatland pull --ff-only")
os.chdir("/content/Peatland")
print(subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout)

flag = "--ignore-requires-python" if sys.version_info < (3, 13) else ""
rc = sh(f"pip install -q {flag} -e ./ml 'segmentation-models-pytorch>=0.5'")
if rc != 0:
    print("Full install failed; falling back to --no-deps.")
    assert sh(f"pip install -q {flag} --no-deps -e ./ml") == 0
    assert (
        sh(
            "pip install -q 'segmentation-models-pytorch>=0.5' geopandas rasterio pyyaml "
            "jsonschema scikit-learn joblib mlflow"
        )
        == 0
    )

import importlib.metadata as importlib_metadata
import numpy

if numpy.__version__ != importlib_metadata.version("numpy"):
    print(
        "RESTART NEEDED: numpy was upgraded. Runtime > Restart session, then rerun from section 2."
    )
else:
    import ml.train, ml.model  # noqa: F401

    print("ml.train imports fine; torch", torch.__version__)

## 4. Copy the data to local disk

Reading COG windows from the Drive mount is slow, so unpack to `/content/data`.

In [ ]:
import json, os, zipfile

ZIP = f"{DRIVE_DIR}/peatwatch_data.zip"
DATA = "/content/data"
assert os.path.isfile(ZIP), (
    f"Upload peatwatch_data.zip to {DRIVE_DIR} first (see docs/runbooks/colab.md)."
)
if not os.path.isdir(f"{DATA}/composites"):
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(DATA)
for sub in ("composites", "labels"):
    for f in sorted(os.listdir(f"{DATA}/{sub}")):
        print(f"{sub}/{f}", round(os.path.getsize(f"{DATA}/{sub}/{f}") / 1e6, 1), "MB")
manifest = json.load(open(f"{DATA}/MANIFEST.json"))
print("fetched/updated:", manifest.get("updated"), "code:", manifest.get("code_version"))
for c in manifest.get("caveats", []):
    print("-", c)

## 5. Colab config

A copy of `ml/configs/unet.yaml` with the Colab changes: ImageNet encoder weights (smp adapts conv1 to 18 bands), CUDA, output on Drive, and one `data.sources` entry per composite, paired with `labels/<region>.tif` (the part of the file name before `__`).

In [ ]:
import glob, yaml

cfg = yaml.safe_load(open("/content/Peatland/ml/configs/unet.yaml"))

sources = []
for comp in sorted(glob.glob(f"{DATA}/composites/*.tif")):
    region = os.path.basename(comp).split("__")[0]
    lab = f"{DATA}/labels/{region}.tif"
    assert os.path.isfile(lab), f"no label raster for {comp}: expected {lab}"
    sources.append({"composite": comp, "labels": lab, "label_class_column": "class"})
assert sources, "no composites found"
cfg["data"]["sources"] = sources
cfg["data"]["patch_size"] = PATCH_SIZE
cfg["data"]["stride"] = STRIDE
cfg["model"]["encoder_weights"] = "imagenet"
cfg["train"].update(
    {"device": "cuda", "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "amp": True}
)
if EPOCHS is not None:
    cfg["train"]["epochs"] = EPOCHS
cfg["output"] = {"dir": f"{DRIVE_DIR}/unet", "resume": True}
cfg["mlflow"]["enabled"] = USE_MLFLOW
if USE_MLFLOW:
    cfg["mlflow"]["tracking_uri"] = "sqlite:////content/mlflow.db"  # local; lost with the session

CONFIG = "/content/unet_colab.yaml"
yaml.safe_dump(cfg, open(CONFIG, "w"), sort_keys=False)
print(open(CONFIG).read())

## 6. Sanity check: split sizes and class counts

Reads only the label rasters, not the composites. Check that train has all four classes (bog is rare, a few percent) and that test is not empty. The Glen Trool demo box is forced into test and must never be in train.

In [ ]:
import numpy as np
from ml.train import config_from_dict, prepare_data

prep = prepare_data(config_from_dict(yaml.safe_load(open(CONFIG))).data)
print("patch counts (incl. dropped buffer patches):", prep.split.counts())
print("train patches used:", len(prep.train_items), " val patches used:", len(prep.val_items))

NAMES = ["other", "bog", "woodland", "felled_or_burnt_bare"]
tot = {s: np.zeros(4, np.int64) for s in ("train", "val", "test")}
for si, (ds, off) in enumerate(zip(prep.raw, prep.offsets)):
    for pi in range(len(ds)):
        s = prep.split.patch_split[off + pi]
        if s is None:
            continue
        r, c = ds.origins[pi]
        lab = ds.labels[r : r + ds.size, c : c + ds.size]
        tot[s] += np.bincount(lab[lab < 4].ravel(), minlength=4)[:4]
print(f"{'split':6}" + "".join(f"{n:>22}" for n in NAMES) + f"{'total':>14}")
for s, v in tot.items():
    print(f"{s:6}" + "".join(f"{x:>22,}" for x in v) + f"{v.sum():>14,}")
    print(f"{'':6}" + "".join(f"{(x / max(v.sum(), 1)):>22.1%}" for x in v))
assert (tot["train"] > 0).all(), "a class is missing from train; do not train, rebuild the data"
assert tot["test"].sum() > 0, "no test patches"

## 7. Train

Output is tee'd to `train.log` on Drive. Expect roughly 10 to 30 minutes on a T4 for ~300 patches and 50 epochs (an estimate, not measured). The first epoch is slower because it fits the band statistics and downloads the ResNet34 weights.

If the session disconnects: reconnect, rerun sections 1 to 5 (the clone, install and unzip are repeated; outputs on Drive are kept), then run this cell again. It continues from `last.pt` with the same split (it refuses to resume if the split changed). To train longer, set `EPOCHS` in section 1, rerun section 5 and this cell.

In [ ]:
t0 = time.time()
rc = subprocess.run(
    f"cd /content/Peatland && python -u -m ml.train --config {CONFIG} 2>&1 | tee -a {DRIVE_DIR}/train.log",
    shell=True,
).returncode
print("exit code", rc, "elapsed min", round((time.time() - t0) / 60, 1))

## 8. Check the artifact on CPU and list the files

In [ ]:
OUT = f"{DRIVE_DIR}/unet"
for f in sorted(os.listdir(OUT)):
    p = f"{OUT}/{f}"
    print(f, round(os.path.getsize(p) / 1e6, 2) if os.path.isfile(p) else "dir", "MB")
print(json.load(open(f"{OUT}/config.json"))["model"])
# CUDA hidden on purpose: this is the CPU load check the API worker will do.
r = subprocess.run(
    f"cd /content/Peatland && CUDA_VISIBLE_DEVICES= python scripts/check_artifact.py {OUT}/best.pt",
    shell=True,
    capture_output=True,
    text=True,
)
print(r.stdout, r.stderr)
assert r.returncode == 0, "best.pt does not load on CPU"
print(
    subprocess.run(
        f"tail -n 8 {DRIVE_DIR}/train.log", shell=True, capture_output=True, text=True
    ).stdout
)

## 9. Download

The files are already on Drive in `MyDrive/peatwatch/unet/` (you can also download them from drive.google.com). This cell zips the three that matter and offers a download. Put them in `artifacts/unet/` in your local repo (see the runbook).

In [ ]:
import shutil
from google.colab import files

bundle = "/content/unet_artifacts"
os.makedirs(bundle, exist_ok=True)
for f in ("best.pt", "split.json", "config.json"):
    shutil.copy(f"{OUT}/{f}", bundle)
shutil.copy(f"{DATA}/MANIFEST.json", bundle)
shutil.copy(f"{DRIVE_DIR}/train.log", bundle)
shutil.make_archive("/content/unet_artifacts", "zip", bundle)
files.download("/content/unet_artifacts.zip")